# 混合模式散射参数转换在分析差分器件时，通常使用混合模式散射参数来分析差分模式和共模特性。Scikit-rf 提供了函数，以便于在单端和混合模式散射参数之间进行转换。尽管如此，用户必须小心地正确设置端口，并注意混合模式矩阵的形式，以防止混淆。本笔记本将介绍从单端散射参数转换为混合模式散射参数的过程，并以 [50 欧姆校准负载](https://www.formfactor.com/download/iss-map-129-246/) 作为示例。实验端口设置包括两个差分探头，如图所示：<img src="mixedmodebasics_files/setup.PNG">将以正常“单端”模式下对 50 欧姆负载进行测量的实验数据与保存为混合模式格式并通过 Keysight 集成的 True Mode Stimulus 测量的实验数据进行比较，该 Stimulus 在测量过程中施加真实的差分和共模信号。

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np

import skrf as rf

sedatafile = r'mixedmodebasics_files/load_se.s4p'
mmdatafile = r'mixedmodebasics_files/load_truemode_balbal.s4p'

for file in [sedatafile, mmdatafile]:
    with open(file, encoding='cp1252') as f:
        for line in f:
            print(line.rstrip())
            if re.search('#', line):
                break
            else:
                pass
    print('\n')

混合模式数据的头部数据表明，它以以下格式保存：$$\begin{bmatrix}\begin{bmatrix}    S_{dd} & S_{dc} \\    S_{cd} & S_{cc}\end{bmatrix}_{11} &\begin{bmatrix}    S_{dd} & S_{dc} \\    S_{cd} & S_{cc}\end{bmatrix}_{12} \\\begin{bmatrix}    S_{dd} & S_{dc} \\    S_{cd} & S_{cc}\end{bmatrix}_{21} &\begin{bmatrix}    S_{dd} & S_{dc} \\    S_{cd} & S_{cc}\end{bmatrix}_{22}\end{bmatrix}$$重要的是要记住这一点，因为这种格式可能会因不同的软件和硬件而异。例如，当存在两个平衡端口时，skrf会将单端数据转换为以下形式：$$\begin{bmatrix}\begin{bmatrix}    S_{11} & S_{12} \\    S_{21} & S_{22}\end{bmatrix}_{dd} &\begin{bmatrix}    S_{11} & S_{12} \\    S_{21} & S_{22}\end{bmatrix}_{dc} \\\begin{bmatrix}    S_{11} & S_{12} \\    S_{21} & S_{22}\end{bmatrix}_{cd} &\begin{bmatrix}    S_{11} & S_{12} \\    S_{21} & S_{22}\end{bmatrix}_{cc}\end{bmatrix}$$为了转换我们的单端数据，我们首先需要将端口配对，使其与实验设置期间的原始端口对应，其中端口 1 和 3 组成一个平衡端口，2 和 4 组成另一个探头。然后，我们可以使用 skrf.Network 类的 `se2gmm()` 方法来转换为混合模式的散射参数矩阵，其中 `p` 参数用于指定混合模式端口的数量。Skrf 将以最低编号的端口（在我们的重新编号之后为 1 和 3）开始成对地转换端口，并继续进行，直到矩阵包含 `p` 个混合模式端口，剩余的端口将保持为单端。

In [ ]:
sedata = rf.Network(sedatafile)
sedata.renumber([0, 1, 2, 3], [0, 2, 1, 3])  # pair ports as 1,3 and 2,4 to match experimental setup
sedata.se2gmm(p=2)  # two balanced ports
# sedata now in form  Sdd  Sdc with each submatrix as S11 S12
#                     Scd  Scc                        S21 S22

以下函数将 skrf 格式的二端口平衡网络转换为 Keysight 使用的格式，以便更轻松地进行数据比较：

In [ ]:
def gmm_reorder(m):
    """
    Reorders data from form 11 12 with each submatrix as dd dc
                            21 22                        cd cc
    to form dd dc with each submatrix as 11 12
            cd cc                        21 22
    """
    b = np.array([1, 0, 0, 0,
                  0, 0, 1, 0,
                  0, 1, 0, 0,
                  0, 0, 0, 1]).reshape(4, 4)
    m = b.dot(m.dot(b))
    return m

mmdata = rf.Network(mmdatafile)
# raw data is mmdata in form S11 S12 with each submatrix as dd dc
#                            S21 S22                        cd cc
for i in range(mmdata.frequency.npoints):
    mmdata.s[i, :, :] = gmm_reorder(mmdata.s[i, :, :])
mmdata.port_modes = ['D', 'D', 'C', 'C']

现在，这两个网络都具有相同的格式，我们可以看到 skrf 认为它们并不相等。这是因为这两个网络并非完全相同，它们只是对同一设备的两次测量结果，并且都包含实验噪声和误差。如果放宽比较的容差，那么对这两个网络进行比较的结果将返回 True：

In [ ]:
print(sedata == mmdata) # uses np.allclose with tight tolerances
print(np.allclose(abs(sedata.s), abs(mmdata.s), rtol=1, atol=1e-3)) # relaxed tolerances

绘制误差图后，误差看起来像这样：

In [ ]:
for m in range(4):
    for n in range(4):
        plt.plot(sedata.f, abs(mmdata.s)[:,m,n]-abs(sedata.s)[:,m,n], label=f'S{sedata._fmt_trace_name(m, n)}')
        plt.title('Magnitude Error between Measurements')
        plt.legend(bbox_to_anchor=(1.1, 1.05))



In [ ]:
fig, axes = plt.subplots(2,2, sharex=True, sharey='row', figsize=(14,6))
sedata.plot_s_db(ax=axes[0][0])
mmdata.plot_s_db(ax=axes[0][1])
sedata.plot_s_deg(ax=axes[1][0])
mmdata.plot_s_deg(ax=axes[1][1])
axes[0][0].get_legend().remove()
axes[1][0].get_legend().remove()
axes[1][1].get_legend().remove()
axes[0][1].get_legend().remove()
axes[0][0].set_title('sedata Magnitude')
axes[0][1].set_title('mmdata Magnitude')
axes[1][0].set_title('sedata Phase')
axes[1][1].set_title('mmdata Phase')

fig.legend(*axes[0, 1].get_legend_handles_labels(), loc="center right")
fig.tight_layout()
plt.subplots_adjust(top=0.9, right=0.8)



In [ ]:
fig, axes = plt.subplots(4,4, sharex=True, figsize=(14,8))
for m in range(4):
    for n in range(4):
        sedata.plot_s_deg_unwrap(m=m, n=n, ax=axes[m][n])
        mmdata.plot_s_deg_unwrap(m=m, n=n, ax=axes[m][n], ls='--')
        axes[m][n].get_legend().remove()
        axes[m][n].set_title(f'S{sedata._fmt_trace_name(m, n)}')
fig.tight_layout()